# Conversation Agent Demo

This notebook shows a minimal conversation agent that uses only working memory.

It uses:
- `BaseAgent`
- `WorkingMemory`
- `MemoryRetrieveNode`
- one custom `ConversationNode(PlannerNode)`
- `ResponseNode`
- `WhatsAppNode`
- `MemoryNode`
- `JSONLTraceSink`
- a local Qwen model for the general conversation path

There are no tools, no ReAct loop, and no long-term memory layers in this example. Outbound delivery happens through `WhatsAppNode` inside the graph.


In [1]:
import os
import re
from pathlib import Path
from types import SimpleNamespace
from langgraph.graph import END, START, StateGraph


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current notebook path.")


def load_env_file(dotenv_path: Path) -> None:
    if not dotenv_path.exists():
        return
    for raw_line in dotenv_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip("'").strip('"'))


repo_root = find_repo_root(Path.cwd())
load_env_file(repo_root / ".env")
from src.agents.base_agent import BaseAgent
from src.agents.nodes import MemoryNode, MemoryRetrieveNode, PlannerNode, ResponseNode, WhatsAppNode
from src.agents.nodes.types import AgentState
from src.interfaces.whatsapp import MockWhatsAppInterface, TwilioWhatsAppInterface, WhatsAppInterface
from src.mailmind.config import AppSettings
from src.llm.qwen import Qwen3_1_7BLLM
from src.memory import WorkingMemory
from src.platform_logging.tracing import ExecutionTrace, JSONLTraceSink, trace_turn
from src.tools.registry import ToolRegistry


class ConversationNode(PlannerNode):
    def plan(
        self,
        *,
        user_input: str,
        memory=None,
        observation=None,
        memory_context=None,
        system_prompt=None,
        user_prompt=None,
        available_tools=None,
    ):
        del observation, available_tools
        assert memory is not None
        working_memory = (memory_context or {}).get("working", memory)

        match = re.search(r"my name is\s+([A-Za-z][A-Za-z\s'-]*)", user_input, re.IGNORECASE)
        if match:
            name = match.group(1).strip()
            return SimpleNamespace(
                thought="Stored the user name.",
                tool_call=None,
                memory_updates=[{"target": "working", "operation": "set_state", "values": {"user_name": name}}],
                respond_directly=True,
                response_text="I'll remember that.",
                done=True,
            )

        if "what is my name" in user_input.lower() or "what's my name" in user_input.lower():
            name = working_memory.state.get("user_name")
            if name:
                return SimpleNamespace(
                    thought="Answered from working memory.",
                    tool_call=None,
                    respond_directly=True,
                    response_text=f"Your name is {name}.",
                    done=True,
                )
            return SimpleNamespace(
                thought="No name stored in working memory.",
                tool_call=None,
                respond_directly=True,
                response_text="You have not told me your name yet.",
                done=True,
            )

        return super().plan(
            user_input=user_input,
            memory=memory,
            system_prompt=system_prompt,
            user_prompt=user_prompt,
        )


class ConversationAgent(BaseAgent):
    def __init__(self, *, llm, memory: WorkingMemory, whatsapp, trace_sink=None) -> None:
        super().__init__(llm=llm, agent_name="conversation_agent", logger=None, trace_sink=trace_sink)
        self.memory = memory
        self.whatsapp = whatsapp
        self.memory_retrieve_node = MemoryRetrieveNode(
            tool_registry=ToolRegistry(),
            memories=[WorkingMemory],
        )
        self.conversation_node = ConversationNode(
            llm=llm,
            system_prompt="You are a concise conversation assistant.",
            user_prompt="User: {user_input}",
        )
        self.response_node = ResponseNode(default_response="I do not know how to respond.")
        self.whatsapp_node = WhatsAppNode(interface=whatsapp)
        self.memory_node = MemoryNode(memories=[memory])
        graph = StateGraph(AgentState)
        graph.add_node("retrieve_memory", self.memory_retrieve_node.execute)
        graph.add_node("plan", self.conversation_node.execute)
        graph.add_node("respond", self.response_node.execute)
        graph.add_node("whatsapp", self.whatsapp_node.execute)
        graph.add_node("memory", self.memory_node.execute)
        graph.add_edge(START, "retrieve_memory")
        graph.add_edge("retrieve_memory", "plan")
        graph.add_conditional_edges(
            "plan",
            self.conversation_node.route,
            {"respond": "respond", "end": END, "act": "respond"},
        )
        graph.add_edge("respond", "whatsapp")
        graph.add_edge("whatsapp", "memory")
        graph.add_edge("memory", END)
        self.graph = graph.compile()

    def run(self, user_input: str, session_id: str | None = None) -> str:
        session_key = session_id or self.memory.session_id
        trace = ExecutionTrace(agent_name=self.agent_name, session_id=session_key, user_input=user_input)
        with trace_turn(trace, sink=self.trace_sink):
            state = self.graph.invoke(
                {
                    "session_id": session_key,
                    "user_input": user_input,
                    "memory": self.memory,
                    "steps": 0,
                }
            )
            trace.finish(status="completed")
        return state["response"]


def build_whatsapp_interface(settings: AppSettings) -> WhatsAppInterface:
    if settings.whatsapp_mode == "fake":
        return MockWhatsAppInterface()
    return TwilioWhatsAppInterface(
        account_sid=settings.integrations.twilio_account_sid,
        auth_token=settings.integrations.twilio_auth_token,
        whatsapp_from=settings.integrations.twilio_whatsapp_from,
    )


settings = AppSettings.from_env()
print("Loaded .env from:", repo_root / ".env")
qwen_llm = Qwen3_1_7BLLM(model_name="Qwen/Qwen3-1.7B", max_new_tokens=128, enable_thinking=False)
memory = WorkingMemory(session_id=settings.notification_destination or "whatsapp:+919999999999")
trace_sink = JSONLTraceSink(repo_root / "data" / "logs" / "conversation_agent_trace.jsonl")
whatsapp = build_whatsapp_interface(settings)
agent = ConversationAgent(llm=qwen_llm, memory=memory, whatsapp=whatsapp, trace_sink=trace_sink)
print(f"WhatsApp mode: {settings.whatsapp_mode}")
print(f"Session destination: {memory.session_id}")


Loaded .env from: /Users/saketm10/Projects/openclaw_agents/.env
WhatsApp mode: twilio
Session destination: whatsapp:+918895551384


In [2]:
agent.run("My name is Saket", session_id=memory.session_id)


"I'll remember that."

In [3]:
agent.run("What is my name?", session_id=memory.session_id)


'Your name is Saket.'

In [4]:
type(whatsapp).__name__


'TwilioWhatsAppInterface'

In [5]:
memory.state


{'node_llms': {'planner': 'Qwen/Qwen3-1.7B'},
 'current_goal': 'What is my name?',
 'user_name': 'Saket'}

In [6]:
if hasattr(whatsapp, "outbound_messages"):
    print(whatsapp.outbound_messages)
else:
    print({"transport": type(whatsapp).__name__, "destination": memory.session_id, "sent_via_twilio": True})


{'transport': 'TwilioWhatsAppInterface', 'destination': 'whatsapp:+918895551384', 'sent_via_twilio': True}


In [7]:
[(item["role"], item["content"]) for item in memory.recent_items]


[('user', 'My name is Saket'),
 ('agent', "I'll remember that."),
 ('user', 'What is my name?'),
 ('agent', 'Your name is Saket.'),
 ('user', 'What is my name?'),
 ('agent', 'Your name is Saket.')]

In [8]:
'''from twilio.rest import Client

client = Client(settings.integrations.twilio_account_sid, settings.integrations.twilio_auth_token)
message = client.messages.create(
    from_=settings.integrations.twilio_whatsapp_from,
    content_sid="HXb5b62575e6e4ff6129ad7c8efe1f983e",
    content_variables="{\"1\":\"12/1\",\"2\":\"3pm\"}",
    to=memory.session_id,
)

{
    "sid": message.sid,
    "status": str(message.status),
    "to": message.to,
    "from": message.from_,
    "error_code": message.error_code,
    "error_message": message.error_message,
}'''


'from twilio.rest import Client\n\nclient = Client(settings.integrations.twilio_account_sid, settings.integrations.twilio_auth_token)\nmessage = client.messages.create(\n    from_=settings.integrations.twilio_whatsapp_from,\n    content_sid="HXb5b62575e6e4ff6129ad7c8efe1f983e",\n    content_variables="{"1":"12/1","2":"3pm"}",\n    to=memory.session_id,\n)\n\n{\n    "sid": message.sid,\n    "status": str(message.status),\n    "to": message.to,\n    "from": message.from_,\n    "error_code": message.error_code,\n    "error_message": message.error_message,\n}'